# Notebook 3: Master Interactive Suite — All 6 OLS Assumptions & Simulations
**Prepared for:** Assumptions of Least Square Regressions Lecture Suite  
**Author:** Nick Lim

This master interactive Jupyter notebook synthesizes simulation exercises for **all 6 Ordinary Least Squares (OLS) assumptions** covered in our lecture suite. Run each code cell sequentially to observe data generation, parameter estimation, failure modes when assumptions are broken, and mathematical fixes in action.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import scipy.stats as stats

# Polyfill for matplotlib compatibility
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = lambda key: matplotlib.rcParams[key]

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 12

---  
## 1. Linearity in Parameters (Assumption 1)
### When Forcing a Straight Line Through Curved Data Fails

We generate $n=60$ observations from a cubic relationship: $Y_i = -5 + X_i^3 + \epsilon_i$. Notice how fitting a standard linear model ($\hat{Y} = \hat{\beta}_0 + \hat{\beta}_1 X$) results in severe underfitting and systematic residual curvature. Transforming the predictor to $X^3$ restores exact linearity in parameters.

In [ ]:
n = 60
x = np.linspace(0, 5, n)
y_cubic = -5.0 + (x ** 3) + np.random.normal(0, 7.5, n)

# Linear OLS Fit
X_lin = sm.add_constant(x)
mod_lin = sm.OLS(y_cubic, X_lin).fit()

# Transformed OLS Fit (X^3)
x_cube = x ** 3
X_trans = sm.add_constant(x_cube)
mod_trans = sm.OLS(y_cubic, X_trans).fit()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Bad Linear Fit
axes[0].scatter(x, y_cubic, color="#f59e0b", alpha=0.85, label="Curved Data ($Y = -5 + X^3 + \epsilon$)")
axes[0].plot(x, mod_lin.predict(X_lin), color="#dc2626", linewidth=3, label="Linear OLS Fit (Underfitting & Biased)")
axes[0].set_title("Violation: Forcing Linear Fit Through Cubic Data")
axes[0].set_xlabel("Predictor ($X$)")
axes[0].set_ylabel("Outcome ($Y$)")
axes[0].legend()

# Right: Transformed Fit
axes[1].scatter(x_cube, y_cubic, color="#16a34a", alpha=0.85, label="Data vs. Transformed Predictor ($X^3$)")
sort_idx = np.argsort(x_cube)
axes[1].plot(x_cube[sort_idx], mod_trans.predict(X_trans)[sort_idx], color="#1e293b", linewidth=3, label=f"Transformed OLS: $\hat{{Y}} = {mod_trans.params[0]:.1f} + {mod_trans.params[1]:.2f}X^3$")
axes[1].set_title("The Fix: Transformation ($X^3$) Restores Linearity")
axes[1].set_xlabel("Transformed Predictor ($X^3$)")
axes[1].legend()

plt.tight_layout()
plt.show()

---  
## 2. Representative Samples Across Ranges (Assumption 2)
### Why Range Subsetting Reverses Regression Slopes

When sample selection is truncated to narrow sub-ranges ($0<X<1.4$, $2.2<X<3.4$, $4.2<X<5.4$), local regressions can yield **negative** slopes ($\hat{\beta}_1 \approx -0.80$) even though the global population trend is strongly **positive** across the full range!

In [ ]:
n_sub = 45
x_low = np.random.uniform(0.2, 1.4, n_sub)
x_mid = np.random.uniform(2.2, 3.4, n_sub)
x_high = np.random.uniform(4.2, 5.4, n_sub)

y_low = 1.5 - 0.8 * (x_low - 0.8) + np.random.normal(0, 0.6, n_sub)
y_mid = 4.5 - 0.8 * (x_mid - 2.8) + np.random.normal(0, 0.6, n_sub)
y_high = 7.5 - 0.8 * (x_high - 4.8) + np.random.normal(0, 0.6, n_sub)

x_all = np.concatenate([x_low, x_mid, x_high])
y_all = np.concatenate([y_low, y_mid, y_high])

mod_overall = sm.OLS(y_all, sm.add_constant(x_all)).fit()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x_low, y_low, color="#f59e0b", s=60, alpha=0.85, label="Lower 3rd Sample")
ax.scatter(x_mid, y_mid, color="#16a34a", s=60, alpha=0.85, label="Middle 3rd Sample")
ax.scatter(x_high, y_high, color="#f97316", s=60, alpha=0.85, label="Upper 3rd Sample")

x_grid = np.linspace(0, 5.6, 100)
ax.plot(x_grid, mod_overall.predict(sm.add_constant(x_grid)), color="#1e293b", linewidth=3.5, label=f"Overall Population OLS Trend ($\\hat{{\\beta}}_1 = +{mod_overall.params[1]:.2f}$)")

for xs, ys, col, name in [(x_low, y_low, "#f59e0b", "Lower 3rd OLS"),
                          (x_mid, y_mid, "#16a34a", "Middle 3rd OLS"),
                          (x_high, y_high, "#f97316", "Upper 3rd OLS")]:
    m_sub = sm.OLS(ys, sm.add_constant(xs)).fit()
    s_grid = np.linspace(np.min(xs)-0.1, np.max(xs)+0.1, 20)
    ax.plot(s_grid, m_sub.predict(sm.add_constant(s_grid)), color=col, linewidth=2.8, linestyle="--", label=f"{name} ($\\hat{{\\beta}}_1 = {m_sub.params[1]:.2f}$)")

ax.set_title("Range Subsetting Reversal: Local Negative Slopes vs Global Positive Trend")
ax.set_xlabel("Predictor ($X$)")
ax.set_ylabel("Outcome ($Y$)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

---  
## 3. Zero Conditional Mean ($E[\epsilon|X] = 0$) & Exogeneity (Assumption 3)
### Step-by-Step Intercept Shifts and Slope Tilts

When omitted variables, measurement errors, or simultaneity cause residuals $\epsilon_i$ to correlate with $X$, the conditional expectation $E[\epsilon|X] \neq 0$. Let's examine how shifting the intercept ($\pm 1$) or tilting the slope ($1.2X - 0.5$, $0.8X + 0.5$) systematically biases predictions and residual expectations.

In [ ]:
n = 100
x_ex = np.random.uniform(0, 5, n)
y_ex = x_ex + np.random.normal(0, 1.0, n)
xg = np.linspace(0, 5, 100)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x_ex, y_ex, color="#f59e0b", s=55, alpha=0.75, label="Observed Data ($Y = X + \epsilon$)")
ax.plot(xg, xg, color="#16a34a", linewidth=3.5, label="True OLS Relationship ($Y = X$)")
ax.plot(xg, xg + 1, color="#eab308", linewidth=2.5, linestyle="--", label="Step 1: Intercept Shift Up ($Y = X + 1 \implies E[e|X] = -1$)")
ax.plot(xg, xg - 1, color="#dc2626", linewidth=2.5, linestyle="--", label="Step 2: Intercept Shift Down ($Y = X - 1 \implies E[e|X] = +1$)")
ax.plot(xg, 1.2 * xg - 0.5, color="#f97316", linewidth=2.5, linestyle="-.", label="Step 3: Steep Slope Bias ($Y = 1.2X - 0.5$)")
ax.plot(xg, 0.8 * xg + 0.5, color="#1e293b", linewidth=2.5, linestyle="-.", label="Step 4: Shallow Slope Bias ($Y = 0.8X + 0.5$)")

ax.set_title("Exogeneity Violation: Overview of Candidate Lines and Residual Bias")
ax.set_xlabel("Predictor ($X$)")
ax.set_ylabel("Outcome ($Y$)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

---  
## 4. Error Variance: Homoscedasticity vs. Megaphone Heteroscedasticity & Autocorrelation (Assumption 4)

We compare:
1. **Homoscedasticity:** Constant error variance ($\sigma = 0.6$), parallel envelope lines.
2. **Heteroscedasticity:** Expanding megaphone variance ($\sigma_i = 0.45X_i$).
3. **Temporal Autocorrelation:** Order-dependent drift ($e_t = \mathcal{N}(0,1) + t/10 - 7.5$).

In [ ]:
n = 150
x_var = np.random.uniform(1, 10, n)

# 1. Homoscedastic
y_ho = 3.0 + 2.0 * x_var + np.random.normal(0, 0.6, n)
m_ho = sm.OLS(y_ho, sm.add_constant(x_var)).fit()

# 2. Heteroscedastic
y_he = 3.0 + 2.0 * x_var + np.random.normal(0, 0.45 * x_var, n)
m_he = sm.OLS(y_he, sm.add_constant(x_var)).fit()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Homoscedastic plot
axes[0].scatter(m_ho.fittedvalues, m_ho.resid, color="#f59e0b", alpha=0.85)
axes[0].axhline(0, color="#1e293b", linestyle="--", linewidth=2)
axes[0].axhline(1.96*0.6, color="#eab308", linestyle=":", linewidth=2.5, label="Parallel Envelope Shape ($\pm 1.96\sigma$)")
axes[0].axhline(-1.96*0.6, color="#eab308", linestyle=":", linewidth=2.5)
axes[0].set_title("Homoscedasticity: Uniform Parallel Envelope")
axes[0].set_xlabel("Fitted Values ($\\hat{Y}$)")
axes[0].set_ylabel("Residuals ($e_i$)")
axes[0].legend()
axes[0].set_ylim(-2.2, 2.2)

# Heteroscedastic plot
axes[1].scatter(m_he.fittedvalues, m_he.resid, color="#dc2626", alpha=0.8)
axes[1].axhline(0, color="#1e293b", linestyle="--", linewidth=2)
x_env = np.linspace(np.min(m_he.fittedvalues), np.max(m_he.fittedvalues), 50)
axes[1].plot(x_env, 0.38*(x_env-np.min(x_env))+0.5, color="#f97316", linestyle=":", linewidth=2.8, label="Megaphone Outer Envelope Shape")
axes[1].plot(x_env, -0.38*(x_env-np.min(x_env))-0.5, color="#f97316", linestyle=":", linewidth=2.8)
axes[1].set_title("Heteroscedasticity: Expanding Megaphone Envelope")
axes[1].set_xlabel("Fitted Values ($\\hat{Y}$)")
axes[1].set_ylabel("Residuals ($e_i$)")
axes[1].legend()

plt.tight_layout()
plt.show()

# 3. Temporal Autocorrelation Drift
time_idx = np.arange(1, n + 1)
errors_temporal = np.random.normal(0, 1.0, n) + (time_idx / 10.0) - 7.5
plt.figure(figsize=(9, 4.5))
plt.plot(time_idx, errors_temporal, color="#f97316", marker='o', markersize=4, linewidth=1.8, alpha=0.85)
plt.axhline(0, color="#1e293b", linewidth=2, linestyle="--")
plt.title("Order Independence Violation: Temporal Residual Drift ($e_t = \\mathcal{N}(0,1) + t/10$)")
plt.xlabel("Observation Order / Time Index ($t$)")
plt.ylabel("Residuals ($e_t$)")
plt.show()

---  
## 5. Normal Q-Q Plot Diagnostic ($n \le 30$) & Outliers/Leverage (Assumption 5)
### Cook's Distance ($D_i$) and the Magnetic Pull of Extreme Leverage Outliers

We compare a clean regression against a dataset contaminated by a single extreme horizontal leverage outlier ($X=12.0, Y=-3.0$). Notice how Cook's distance flags this point ($D_i > 4/n$) and how one point tilts the slope from positive to near-zero!

In [ ]:
n_clean = 14
x_c = np.linspace(1, 8, n_clean)
y_c = 1.0 + 1.2 * x_c + np.random.normal(0, 0.5, n_clean)

# Append extreme leverage outlier
x_out = np.append(x_c, 12.0)
y_out = np.append(y_c, -3.0)

mod_clean = sm.OLS(y_c, sm.add_constant(x_c)).fit()
mod_out = sm.OLS(y_out, sm.add_constant(x_out)).fit()

# Cook's Distance
influence = mod_out.get_influence()
cooks_d = influence.cooks_distance[0]
threshold = 4 / len(x_out)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Regression Line Tilt
axes[0].scatter(x_c, y_c, color="#2563eb", s=60, label="Standard Observations ($n=14$)")
axes[0].scatter([12.0], [-3.0], color="#dc2626", s=200, marker="*", label="Extreme Leverage Outlier ($X=12, Y=-3$)")
xg = np.linspace(0, 13, 100)
axes[0].plot(xg, mod_clean.predict(sm.add_constant(xg)), color="#16a34a", linewidth=3.2, linestyle="--", label=f"Clean OLS (Slope = {mod_clean.params[1]:.2f})")
axes[0].plot(xg, mod_out.predict(sm.add_constant(xg)), color="#dc2626", linewidth=3.8, label=f"Contaminated OLS (Slope = {mod_out.params[1]:.2f})")
axes[0].set_title("Magnetic Pull of Extreme Leverage Outlier")
axes[0].set_xlabel("Predictor ($X$)")
axes[0].set_ylabel("Outcome ($Y$)")
axes[0].legend()

# Right: Cook's Distance Chart
obs_indices = np.arange(1, len(x_out) + 1)
colors = ["#dc2626" if d > threshold else "#2563eb" for d in cooks_d]
axes[1].bar(obs_indices, cooks_d, color=colors, width=0.6)
axes[1].axhline(threshold, color="#dc2626", linestyle="--", linewidth=2, label=f"Critical Threshold ($4/n = {threshold:.2f}$)")
axes[1].set_title("Cook's Distance ($D_i$) Diagnostic")
axes[1].set_xlabel("Observation Index ($i$)")
axes[1].set_ylabel("Cook's Distance ($D_i$)")
axes[1].legend()

plt.tight_layout()
plt.show()

---  
## 6. Summary: All 6 OLS Assumptions & Diagnostics Checklist

| Assumption | Mathematical Condition | Diagnostic Tool | Standard Fix | 
|---|---|---|---|
| **1. Linearity** | Model is linear in $\beta$ | Residuals vs Fitted (`Y-hat`) | Polynomial/log transformations ($X^3, \ln X$) |
| **2. Representative Samples** | Random sampling from population | Range subsampling checks | Avoid selective truncation / weight samples |
| **3. Exogeneity** | $E[\epsilon_i | X_i] = 0$ | Economic domain reasoning | Instrumental variables (2SLS) / include omitted variables |
| **4. Homoscedasticity** | $\text{Var}(\epsilon_i) = \sigma^2$ (Constant) | Scale-Location & Residual plots | White's Robust Standard Errors (`HC3`) |
| **5. Normality / Outliers** | $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$ / low leverage | Normal Q-Q Plot & Cook's $D_i$ | Robust Regression (`MASS::rlm` M-estimation) |
| **6. No Multicollinearity** | $|X'X| \neq 0$, $\text{VIF} < 10$ | Correlation matrix & VIF table | Drop redundant predictors / ridge regression |